# 🧬 Exercícios — Algoritmos Genéticos

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Explore os operadores genéticos e aplique algoritmos evolutivos a problemas de otimização.


## 1. Componentes de um Algoritmo Genético

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

random.seed(42); np.random.seed(42)

# ─── Problema: maximizar f(x) = x*sin(x) em [0, 10] ───

def fitness(x): return x * np.sin(x)

# Codificação binária (8 bits → [0, 10])
def decodificar(cromossomo, xmin=0, xmax=10):
    inteiro = int(''.join(str(b) for b in cromossomo), 2)
    return xmin + inteiro * (xmax - xmin) / (2**len(cromossomo) - 1)

def criar_populacao(tam, n_bits=8):
    return [[random.randint(0,1) for _ in range(n_bits)] for _ in range(tam)]

def selecao_roleta(populacao, fitnesses):
    """Seleção proporcional à aptidão."""
    total = sum(f for f in fitnesses if f > 0)
    if total == 0: return random.choice(populacao)
    probs = [max(0,f)/total for f in fitnesses]
    return populacao[np.random.choice(len(populacao), p=probs)]

def cruzamento_ponto_unico(p1, p2):
    pt = random.randint(1, len(p1)-1)
    return p1[:pt]+p2[pt:], p2[:pt]+p1[pt:]

def mutacao_bit_flip(cromossomo, taxa=0.05):
    return [1-b if random.random()<taxa else b for b in cromossomo]

# ─── Loop Evolutivo ───
TAM_POP = 50; N_BITS = 12; GERACOES = 80

pop = criar_populacao(TAM_POP, N_BITS)
historico_best = []

for gen in range(GERACOES):
    xs = [decodificar(c) for c in pop]
    fits = [fitness(x) for x in xs]
    historico_best.append(max(fits))
    
    nova_pop = []
    # Elitismo
    elite_idx = np.argmax(fits)
    nova_pop.append(pop[elite_idx])
    
    while len(nova_pop) < TAM_POP:
        p1 = selecao_roleta(pop, fits)
        p2 = selecao_roleta(pop, fits)
        f1, f2 = cruzamento_ponto_unico(p1, p2)
        nova_pop += [mutacao_bit_flip(f1), mutacao_bit_flip(f2)]
    
    pop = nova_pop[:TAM_POP]

melhor_x = decodificar(pop[np.argmax([fitness(decodificar(c)) for c in pop])])
print(f"Melhor x encontrado: {melhor_x:.4f}")
print(f"f(x) = {fitness(melhor_x):.4f}")

# Visualização
x_arr = np.linspace(0, 10, 500)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(x_arr, fitness(x_arr)); plt.axvline(melhor_x,color='r',linestyle='--',label=f'x*={melhor_x:.2f}')
plt.title('f(x) = x·sin(x)'); plt.legend(); plt.grid(True)
plt.subplot(1,2,2)
plt.plot(historico_best,'g-'); plt.title('Evolução do Melhor Fitness'); plt.xlabel('Geração'); plt.grid(True)
plt.tight_layout(); plt.show()


### 📝 Exercício 1

Experimente com diferentes **taxas de mutação** (0.001, 0.01, 0.05, 0.2). Para cada taxa, execute o AG por 80 gerações e plote as curvas de convergência. Qual taxa leva à melhor solução?

In [ ]:
taxas = [0.001, 0.01, 0.05, 0.2]
plt.figure(figsize=(10,4))
for taxa in taxas:
    random.seed(42); np.random.seed(42)
    pop = criar_populacao(TAM_POP, N_BITS)
    hist = []
    for gen in range(GERACOES):
        xs = [decodificar(c) for c in pop]; fits = [fitness(x) for x in xs]
        hist.append(max(fits))
        nova_pop = [pop[np.argmax(fits)]]
        while len(nova_pop) < TAM_POP:
            p1=selecao_roleta(pop,fits); p2=selecao_roleta(pop,fits)
            f1,f2=cruzamento_ponto_unico(p1,p2)
            nova_pop+=[mutacao_bit_flip(f1,taxa),mutacao_bit_flip(f2,taxa)]
        pop=nova_pop[:TAM_POP]
    plt.plot(hist,label=f'taxa={taxa}')
plt.title('Impacto da Taxa de Mutação'); plt.xlabel('Geração'); plt.ylabel('Melhor Fitness')
plt.legend(); plt.grid(True); plt.show()


## 2. AG com Codificação Real — Otimização de Funções 2D

In [ ]:
# Minimizar a função de Rastrigin (muitos ótimos locais)
def rastrigin(x, y):
    return 20 + x**2 - 10*np.cos(2*np.pi*x) + y**2 - 10*np.cos(2*np.pi*y)

# Visualizar a função
xr = yr = np.linspace(-5.12, 5.12, 100)
Xr, Yr = np.meshgrid(xr, yr)
Zr = rastrigin(Xr, Yr)
plt.figure(figsize=(8,6))
plt.contourf(Xr, Yr, Zr, levels=30, cmap='viridis')
plt.colorbar(); plt.title('Função de Rastrigin (mínimo global em 0,0)'); plt.show()

# AG com representação real
def ag_real(func, bounds=(-5.12,5.12), tam_pop=80, geracoes=200, taxa_mut=0.1):
    pop = np.random.uniform(bounds[0], bounds[1], (tam_pop, 2))
    hist = []
    for gen in range(geracoes):
        fits = np.array([func(p[0],p[1]) for p in pop])
        hist.append(fits.min())
        # Seleção por torneio
        idx = np.argsort(fits)
        pop = pop[idx]  # elitismo: mantém os melhores
        nova = [pop[0].copy()]  # elitismo
        while len(nova) < tam_pop:
            t1,t2 = random.sample(range(tam_pop//2), 2)
            pai1,pai2 = pop[t1],pop[t2]
            alfa = random.random()
            filho = alfa*pai1 + (1-alfa)*pai2
            filho += np.random.randn(2)*taxa_mut*(1-gen/geracoes)  # mutação decrescente
            filho = np.clip(filho, bounds[0], bounds[1])
            nova.append(filho)
        pop = np.array(nova[:tam_pop])
    return pop[0], hist

melhor, hist_r = ag_real(rastrigin)
print(f"Melhor solução: x={melhor[0]:.4f}, y={melhor[1]:.4f}")
print(f"f(x,y) = {rastrigin(melhor[0],melhor[1]):.6f}  (ótimo global = 0.0)")
plt.figure(figsize=(8,3)); plt.plot(hist_r,'m-'); plt.title('Convergência na Rastrigin'); plt.xlabel('Geração'); plt.grid(True); plt.show()


### 📝 Exercício 2

Applique o AG para minimizar a **função de Ackley** (outro benchmark clássico). O ótimo global também é em (0,0) com valor 0.

In [ ]:
def ackley(x, y, a=20, b=0.2, c=2*np.pi):
    """Função de Ackley — ótimo global em (0,0) = 0."""
    term1 = -a * np.exp(-b * np.sqrt(0.5*(x**2+y**2)))
    term2 = -np.exp(0.5*(np.cos(c*x)+np.cos(c*y)))
    return term1 + term2 + a + np.e

# ✏️ Aplique o ag_real para a função de Ackley:
melhor_a, hist_a = ag_real(ackley, bounds=(-5,5))
print(f"Ackley — Melhor: x={melhor_a[0]:.4f}, y={melhor_a[1]:.4f}, f={ackley(melhor_a[0],melhor_a[1]):.4f}")
plt.figure(figsize=(8,3)); plt.plot(hist_a,'c-'); plt.title('Convergência na Ackley'); plt.xlabel('Geração'); plt.grid(True); plt.show()


## 3. Exercício Final — TSP com AG

Aplique o AG para resolver o **Problema do Caixeiro Viajante** com 10 cidades.

In [ ]:
# 10 cidades com coordenadas aleatórias
np.random.seed(7)
cidades = np.random.rand(10, 2) * 100
n_cidades = len(cidades)

def distancia_total(rota):
    total = 0
    for i in range(len(rota)):
        a, b = rota[i], rota[(i+1)%len(rota)]
        total += np.linalg.norm(cidades[a]-cidades[b])
    return total

def criar_rota(): return list(np.random.permutation(n_cidades))

def crossover_ox(p1, p2):
    """Order Crossover (OX) para permutações."""
    a, b = sorted(random.sample(range(n_cidades), 2))
    segmento = p1[a:b]
    filho = [c for c in p2 if c not in segmento]
    return filho[:a] + segmento + filho[a:]

def mutacao_swap(rota, taxa=0.1):
    rota = rota[:]
    if random.random() < taxa:
        i, j = random.sample(range(n_cidades), 2)
        rota[i], rota[j] = rota[j], rota[i]
    return rota

# AG para TSP
pop_tsp = [criar_rota() for _ in range(100)]
hist_tsp = []

for gen in range(300):
    pop_tsp.sort(key=distancia_total)
    hist_tsp.append(distancia_total(pop_tsp[0]))
    nova_pop = pop_tsp[:5]  # elitismo
    while len(nova_pop) < 100:
        p1, p2 = random.sample(pop_tsp[:30], 2)
        filho = crossover_ox(p1, p2)
        filho = mutacao_swap(filho)
        nova_pop.append(filho)
    pop_tsp = nova_pop[:100]

melhor_rota = pop_tsp[0]
print(f"Melhor rota: {melhor_rota}")
print(f"Distância total: {distancia_total(melhor_rota):.2f}")

fig, axes = plt.subplots(1,2,figsize=(12,4))
# Rota
ax=axes[0]
rota_fechada = melhor_rota + [melhor_rota[0]]
ax.plot(cidades[rota_fechada,0], cidades[rota_fechada,1], 'b-o')
for i, c in enumerate(cidades): ax.annotate(str(i),(c[0]+0.5,c[1]+0.5))
ax.set_title(f'Melhor Rota (dist={distancia_total(melhor_rota):.1f})')
# Convergência
axes[1].plot(hist_tsp,'g-'); axes[1].set_title('Convergência TSP'); axes[1].grid(True)
plt.tight_layout(); plt.show()
